#### Modelling

##### Importing Libraries and Loading Data

In [45]:
import numpy as np
import pandas as pd
import joblib
from pathlib import Path

from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error
from xgboost import XGBRegressor
import joblib

In [2]:
project_root = Path(".").resolve().parent
df = pd.read_parquet(project_root /'data'/'processed'/'london_features.parquet')

##### Temporal Data Split

In [3]:
# Temporal Split: Train up until 2022, Validate on 2023, Test on 2024

train = df[df['year'] <= 2022].copy()
val = df[df['year']  == 2023].copy()
test = df[df['year'] == 2024].copy()

In [4]:
train.shape

(2693933, 24)

In [5]:
val.shape

(66581, 24)

In [6]:
test.shape

(40529, 24)

##### Target Encoding of Postcode Districts

In [7]:
# Target Encoding of Postcode District
# Replace Each Postcode District with the mean log price of its rows. smoothed toward the
# global mean so rare districts aren't trusted too much. Fit on Training Data ONLY; unseen districts
# fall back to the global training mean. We write it as a function because we apply it twice (once for
# the comparison, once for the final model).

def fit_target_encoding(train_df, smooth=20):
    global_mean = train_df['log_price'].mean()
    stats = train_df.groupby('postcode_district')['log_price'].agg(['mean', 'count'])
    encoding = (stats['mean'] * stats['count'] + global_mean * smooth) / (stats['count'] + smooth)

    return encoding, global_mean

In [8]:
def apply_target_encoding(frame, encoding, global_mean):
    return frame['postcode_district'].map(encoding, na_action=None).fillna(global_mean)

In [9]:
encoding, global_mean = fit_target_encoding(train)
for part in (train, val, test):
    part['district_te'] = apply_target_encoding(part, encoding, global_mean)

##### Filling Missing Values

In [10]:
# Imputing the missing values in the two columns with the median values

for col in ['numberrooms', 'age_year']:
    median = train[col].median()
    for part in (train, val, test):
        part[col] = part[col].fillna(median)

##### Price Model Feature Set

In [11]:
price_features = [
    'log_tfarea', 'numberrooms', 'age_year',
    'lat', 'lon', 'zone', 'imd_index', 'log_income', 'dist_station',
    'propertytype_D', 'propertytype_F', 'propertytype_S', 'propertytype_T',
    'duration_F', 'duration_L',
    'year', 'month', 'district_te',
]

In [12]:
X_train, y_train = train[price_features], train['log_price']
X_val, y_val = val[price_features], val['log_price']
X_test, y_test = test[price_features], test['log_price']

##### Ridge

In [13]:
ridge = make_pipeline(StandardScaler(), Ridge(alpha=1))

In [14]:
ridge.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('standardscaler', ...), ('ridge', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"copy copy: bool, default=TrueIf False, try to avoid a copy and do inplace scaling instead.This is not guaranteed to always work inplace; e.g. if the data isnot a NumPy array or scipy.sparse CSR matrix, a copy may still bereturned.",True
,"with_mean with_mean: bool, default=TrueIf True, center the data before scaling.This does not work (and will raise an exception) when attempted onsparse matrices, because centering them entails building a densematrix which in common use cases is likely to be too large to fit inmemory.",True
,"with_std with_std: bool, default=TrueIf True, scale the data to unit variance (or equivalently,unit standard deviation).",True
,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None


In [16]:
ridge_pred = ridge.predict(X_val)

In [29]:
ridge_r2 = r2_score(y_val, ridge_pred)
print(f'Ridge Validation R2 Score (log): {ridge_r2:.2f}')

Ridge Validation R2 Score (log): 0.49


##### Random Forest

In [18]:
rf = RandomForestRegressor(n_estimators=100, max_depth=20, n_jobs=-1, random_state=42)
rf.fit(X_train, y_train)

RandomForest val R²(log) = 0.864


In [28]:
rf_r2 = r2_score(y_val, rf.predict(X_val))
print(f'Random Forest Validation R2 Score (log) = {rf_r2:.2f}')

Random Forest Validation R2 Score (log) = 0.86


##### Gradient Boosting

In [30]:
xgb = XGBRegressor(n_estimators=800, max_depth=8, learning_rate=0.05, tree_method='hist',
                   subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                   random_state=42, n_jobs=-1)
xgb.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [32]:
xgb_r2 = r2_score(y_val, xgb.predict(X_val))
print(f'Gradient Boost Validation R2 Score (log) = {xgb_r2:.2f}')

Gradient Boost Validation R2 Score (log) = 0.87


##### Final Model

In [33]:
final_train = df[df['year'] <= 2023].copy()
final_test = df[df['year'] == 2024].copy()

In [34]:
encoding, global_mean = fit_target_encoding(final_train)
for part in (final_train, final_test):
    part['district_te'] = apply_target_encoding(part, encoding, global_mean)

In [35]:
impute_medians = {}
for col in ['numberrooms', 'age_year']:
    impute_medians[col] = final_train[col].median()
    for part in (final_train, final_test):
        part[col] = part[col].fillna(impute_medians[col])

In [36]:
final_model = XGBRegressor(
    n_estimators=1400, max_depth=11, learning_rate=0.03,
    min_child_weight=30, reg_lambda=1.5, tree_method='hist',
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
)

In [37]:
final_model.fit(final_train[price_features], final_train['log_price'])

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.8
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [38]:
pred_log = final_model.predict(final_test[price_features])
pred_gbp = 10 ** pred_log
true_gbp = 10 ** final_test['log_price']

In [43]:
print(f'R2(log) = {r2_score(final_test["log_price"], pred_log):.2f}')
print(f'R2(raw £) = {r2_score(true_gbp, pred_gbp):.2f}')
print(f'MAE  = £{mean_absolute_error(true_gbp, pred_gbp):,.0f}')
print(f'Median Ansolute Error = £{np.median(np.abs(true_gbp - pred_gbp)):,.0f}')

R2(log) = 0.89
R2(raw £) = 0.88
MAE  = £92,377
Median Ansolute Error = £46,314


In [44]:
# Checking where the model works. We split properties in 10 Price Bands.
# The table shows where the model is most accurate. Errors as a percentage of
# price tells us the model is reliable through the market's main cluster and weaker at the extremes.

res = final_test[['price']].copy()
res['abs_err'] = np.abs(true_gbp - pred_gbp)
res['decile'] = pd.qcut(res['price'], 10, labels=False)

summary = res.groupby('decile').agg(
    median_price=('price', 'median'),
    median_abs_err=('abs_err', 'median'),
)
summary['pct_err'] = (summary['median_abs_err'] / summary['median_price'] * 100).round(1)
print(summary.round(0).to_string())

        median_price  median_abs_err  pct_err
decile                                       
0           245000.0         29465.0     12.0
1           330000.0         28562.0      9.0
2           390000.0         29982.0      8.0
3           440000.0         30988.0      7.0
4           495000.0         38398.0      8.0
5           555000.0         44412.0      8.0
6           625250.0         53378.0      8.0
7           750000.0         71973.0     10.0
8           935000.0        101111.0     11.0
9          1580000.0        213377.0     14.0


In [46]:
# Saving Artifacts

models_dir = project_root/'models'
bundle = {
    'model': final_model,
    'features': price_features,
    'target_encoding': encoding.to_dict(),
    'te_global_mean': global_mean,
    'impute_medians': impute_medians,
}

joblib.dump(bundle, models_dir /'price_model.joblib')

['C:\\Users\\Abdul Qudus\\Documents\\Data Portfolio\\london-property-intelligence\\models\\price_model.joblib']